In [ ]:
# M5: FAILURE CAUSALITY ENGINE
# LLM-powered root cause analysis using:
#   1. Cortex Search (RAG over maintenance logs)
#   2. AI_COMPLETE (structured reasoning with evidence)
#   3. Sensor context + historical pattern matching

import pandas as pd
import numpy as np
import json
from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.sql("USE DATABASE FAILURE_GENOME_DB").collect()
session.sql("USE SCHEMA ML_MODELS").collect()
print("Session ready.")

In [ ]:
# Create a searchable index over maintenance logs + work orders
# This enables RAG: "find similar past failures"

session.sql("""
CREATE OR REPLACE CORTEX SEARCH SERVICE FAILURE_GENOME_DB.ML_MODELS.MAINTENANCE_SEARCH
    ON search_text
    ATTRIBUTES asset_id, wo_type, priority
    WAREHOUSE = COMPUTE_WH
    TARGET_LAG = '48 hours'
AS (
    SELECT 
        ml.log_id AS doc_id,
        ml.asset_id,
        ml.notes_text || ' | Work Order: ' || wo.wo_type || ' | Priority: ' || wo.priority || 
        ' | Asset: ' || am.asset_name || ' (' || am.asset_type || ')' ||
        ' | Date: ' || TO_CHAR(ml.timestamp, 'YYYY-MM-DD') AS search_text,
        wo.wo_type,
        wo.priority,
        ml.timestamp AS log_date
    FROM FAILURE_GENOME_DB.RAW_IT.MAINTENANCE_LOGS ml
    JOIN FAILURE_GENOME_DB.RAW_IT.WORK_ORDERS wo ON ml.wo_id = wo.wo_id
    JOIN FAILURE_GENOME_DB.RAW_OT.ASSET_MASTER am ON ml.asset_id = am.asset_id
)
""").collect()
print("Cortex Search Service 'MAINTENANCE_SEARCH' created.")
print("Indexing maintenance logs for RAG retrieval...")

In [ ]:
# Create ROOT_CAUSE_ANALYSIS as a stored procedure
# Using $$ delimiters to avoid string escaping issues

session.sql("""
CREATE OR REPLACE PROCEDURE FAILURE_GENOME_DB.ML_MODELS.ANALYZE_ROOT_CAUSE(
    p_asset_id VARCHAR,
    p_current_vib_mag FLOAT,
    p_current_temp FLOAT,
    p_current_rpm FLOAT,
    p_current_amps FLOAT,
    p_fatigue_score FLOAT,
    p_predicted_failure_mode VARCHAR,
    p_predicted_rul_hours FLOAT,
    p_degradation_stage INTEGER
)
RETURNS VARIANT
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
PACKAGES = ('snowflake-snowpark-python')
HANDLER = 'run'
EXECUTE AS CALLER
AS
$$
import json, re

def run(session, p_asset_id, p_current_vib_mag, p_current_temp, p_current_rpm,
        p_current_amps, p_fatigue_score, p_predicted_failure_mode,
        p_predicted_rul_hours, p_degradation_stage):

    search_query = str(p_predicted_failure_mode) + " " + str(p_asset_id) + " degradation failure"
    fleet_query = str(p_predicted_failure_mode) + " vibration temperature failure"

    # Retrieve asset-specific maintenance history via Cortex Search
    try:
        search_params = json.dumps({
            "query": search_query,
            "columns": ["search_text", "asset_id", "wo_type"],
            "filter": {"@eq": {"asset_id": str(p_asset_id)}},
            "limit": 5
        })
        safe_params = search_params.replace("'", "''")
        sql = f"SELECT PARSE_JSON(SNOWFLAKE.CORTEX.SEARCH_PREVIEW('FAILURE_GENOME_DB.ML_MODELS.MAINTENANCE_SEARCH', '{safe_params}'))['results'] AS results"
        rows = session.sql(sql).collect()
        results_arr = json.loads(str(rows[0]["RESULTS"]))
        asset_history = " | ".join([r.get("search_text", "") for r in results_arr[:5]])
    except Exception as e:
        asset_history = "No maintenance history found. Error: " + str(e)

    # Retrieve fleet-wide similar failures
    try:
        fleet_params = json.dumps({
            "query": fleet_query,
            "columns": ["search_text", "asset_id", "wo_type"],
            "limit": 5
        })
        safe_fleet = fleet_params.replace("'", "''")
        sql2 = f"SELECT PARSE_JSON(SNOWFLAKE.CORTEX.SEARCH_PREVIEW('FAILURE_GENOME_DB.ML_MODELS.MAINTENANCE_SEARCH', '{safe_fleet}'))['results'] AS results"
        frows = session.sql(sql2).collect()
        fleet_arr = json.loads(str(frows[0]["RESULTS"]))
        fleet_history = " | ".join([r.get("search_text", "") for r in fleet_arr[:5]])
    except Exception as e:
        fleet_history = "No fleet history found. Error: " + str(e)

    # Build the LLM prompt
    lines = [
        "You are an expert reliability engineer performing root cause analysis.",
        "",
        "CURRENT ASSET STATE:",
        "- Asset ID: " + str(p_asset_id),
        "- Vibration: " + str(p_current_vib_mag) + " mm/s",
        "- Temperature: " + str(p_current_temp) + " C",
        "- RPM: " + str(p_current_rpm),
        "- Current: " + str(p_current_amps) + " A",
        "- Fatigue Score: " + str(p_fatigue_score) + " (0=healthy, 1=critical)",
        "- Predicted Failure Mode: " + str(p_predicted_failure_mode),
        "- Predicted RUL: " + str(p_predicted_rul_hours) + " hours",
        "- Degradation Stage: " + str(p_degradation_stage) + "/4",
        "",
        "MAINTENANCE HISTORY FOR THIS ASSET:",
        asset_history,
        "",
        "SIMILAR FAILURES ACROSS FLEET:",
        fleet_history,
        "",
        'Respond ONLY with valid JSON in this exact format:',
        '{"root_cause": "specific mechanical root cause",',
        '"confidence": 0.9,',
        '"failure_mechanism": "bearing_wear or thermal_degradation or imbalance or misalignment or unknown",',
        '"evidence": ["evidence 1", "evidence 2", "evidence 3"],',
        '"contributing_factors": ["factor 1", "factor 2"],',
        '"risk_level": "CRITICAL or HIGH or MEDIUM or LOW",',
        '"recommended_immediate_action": "what to do now",',
        '"similar_past_failure_reference": "reference to similar event"}',
        "",
        "No markdown, no text outside JSON."
    ]
    prompt = chr(10).join(lines)

    safe_prompt = prompt.replace("'", "''")
    llm_sql = f"SELECT SNOWFLAKE.CORTEX.COMPLETE('llama3.3-70b', '{safe_prompt}') AS response"
    llm_rows = session.sql(llm_sql).collect()
    raw = str(llm_rows[0]["RESPONSE"])

    start = raw.find('{')
    end = raw.rfind('}') + 1
    if start != -1 and end > start:
        try:
            return json.loads(raw[start:end])
        except:
            return {"root_cause": raw, "confidence": 0.5, "parse_error": True}
    return {"root_cause": raw, "confidence": 0.5, "no_json": True}
$$
""").collect()
print("ANALYZE_ROOT_CAUSE procedure created (using llama3.3-70b).")

In [ ]:
# Test on ASSET_001 (bearing wear, high degradation)
# Note: ANALYZE_ROOT_CAUSE is a PROCEDURE, so use CALL not SELECT
result = session.sql("""
CALL FAILURE_GENOME_DB.ML_MODELS.ANALYZE_ROOT_CAUSE(
    'ASSET_001',
    12.5,
    78.0,
    1478.0,
    14.2,
    0.82,
    'bearing_wear',
    72.0,
    3
)
""").collect()

raw = result[0][0]
if isinstance(raw, str):
    analysis = json.loads(raw)
else:
    analysis = raw

print("="*70)
print("ROOT CAUSE ANALYSIS - ASSET_001 (Compressor A1)")
print("="*70)
print(json.dumps(analysis, indent=2))

In [ ]:
# Test on ASSET_004 (misalignment)
# ANALYZE_ROOT_CAUSE is a PROCEDURE, use CALL not SELECT
result2 = session.sql("""
CALL FAILURE_GENOME_DB.ML_MODELS.ANALYZE_ROOT_CAUSE(
    'ASSET_004',
    8.2,
    42.0,
    2948.0,
    18.5,
    0.65,
    'misalignment',
    120.0,
    2
)
""").collect()

raw2 = result2[0][0]
if isinstance(raw2, str):
    analysis2 = json.loads(raw2)
else:
    analysis2 = raw2

print("="*70)
print("ROOT CAUSE ANALYSIS - ASSET_004 (Pump P2)")
print("="*70)
print(json.dumps(analysis2, indent=2))

In [ ]:
# Batch root cause analysis for all at-risk assets
# Note: ANALYZE_ROOT_CAUSE is a PROCEDURE, so we can't use it in a VIEW.
# Instead, we loop through at-risk assets and call it for each.

at_risk = session.sql("""
SELECT ahc.asset_id, ahc.asset_name, ahc.health_score,
       ahc.current_vib_magnitude, ahc.current_temperature, 
       ahc.current_rpm, ahc.current_amps
FROM FAILURE_GENOME_DB.CURATED.ASSET_HEALTH_CURRENT ahc
WHERE ahc.health_score < 70
ORDER BY ahc.health_score ASC
""").to_pandas()

print(f"At-risk assets (health < 70): {len(at_risk)}")
print(at_risk[['ASSET_ID', 'ASSET_NAME', 'HEALTH_SCORE']].to_string(index=False))
print("\nTo analyze any asset, run:")
print("  CALL FAILURE_GENOME_DB.ML_MODELS.ANALYZE_ROOT_CAUSE('ASSET_001', vib, temp, rpm, amps, fatigue, mode, rul, stage)")

In [ ]:
print("="*70)
print("M5: FAILURE CAUSALITY ENGINE - COMPLETE")
print("="*70)
print("""
Architecture:
  1. Cortex Search Service: MAINTENANCE_SEARCH (RAG over 26 maintenance logs)
  2. Stored Procedure: ANALYZE_ROOT_CAUSE (structured LLM reasoning with evidence)
  3. Model: llama3.3-70b via AI_COMPLETE

How it works:
  Input: asset_id + current sensor state + model predictions
    v
  Cortex Search: retrieves similar past failures for this asset
  Cortex Search: retrieves similar failures across the fleet
    v
  AI_COMPLETE (llama3.3-70b): structured reasoning with evidence chain
    v
  Output: JSON with root_cause, confidence, evidence, risk_level, action

Demo script:
  Judge asks: 'Why is Compressor A1 failing?'
  Agent calls: ANALYZE_ROOT_CAUSE('ASSET_001', ...)
  Agent responds with specific evidence from maintenance logs.
""")